# tqdm-postfix-metrics — faded example 3: Gradient norm in tqdm postfix during training

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `tqdm-postfix-metrics`. Running the beacon reports progress on the `Logging: tqdm postfix metrics` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: tqdm postfix metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tqdm-postfix-metrics`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tqdm-postfix-metrics"
DD_SUBTOPIC = "Logging: tqdm postfix metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Monitoring gradient norms alongside loss helps catch exploding or vanishing gradients early. After `loss.backward()` and before `optimizer.step()`, you can compute the gradient norm with `t.nn.utils.clip_grad_norm_` (or manually) and include it in the tqdm postfix. Logging it in a sidecar list preserves the history for inspection.

## Faded exercise 3

Implement `faded3_tqdm_grad_norm(n_steps=5)`. Build a 1-parameter model (`w = t.tensor([2.0], requires_grad=True)`), an SGD optimizer (`lr=0.05`), and inputs `x = t.linspace(0, 1, 8)`, `y_target = 3.0 * x`. Wrap `range(n_steps)` in `tqdm(...)` with `desc='GradWatch'`. For each step: forward (`pred = w * x`), loss (`((pred - y_target)**2).mean()`), `loss.backward()`, then compute `grad_norm = w.grad.abs().item()`. Call `pbar.set_postfix(loss=f'{loss.item():.4f}', grad_norm=f'{grad_norm:.4f}')`. Append both values to a sidecar list, then call `optimizer.step()` and `optimizer.zero_grad()`. Return the log.

**Fill in:** The pbar.set_postfix call that sets both loss and grad_norm on the progress bar.

In [ ]:
from tqdm import tqdm
import torch as t

def faded3_tqdm_grad_norm(n_steps=5):
    t.manual_seed(1)
    w = t.tensor([2.0], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=0.05)
    x = t.linspace(0, 1, 8)
    y_target = 3.0 * x
    log = []
    pbar = tqdm(range(n_steps), desc='GradWatch')
    for step in pbar:
        pred = w * x
        loss = ((pred - y_target) ** 2).mean()
        loss.backward()
        grad_norm = w.grad.abs().item()
        pf = dict(loss=f'{loss.item():.4f}', grad_norm=f'{grad_norm:.4f}')
        raise NotImplementedError()  # TODO: The pbar.set_postfix call that sets both loss and grad_norm on the progress bar.
        log.append(pf)
        optimizer.step()
        optimizer.zero_grad()
    return log


def _test():
    import torch as t
    t.manual_seed(1)
    log = faded3_tqdm_grad_norm(n_steps=5)
    assert len(log) == 5
    for entry in log:
        assert 'loss' in entry
        assert 'grad_norm' in entry
    # First entry should have non-zero grad norm
    assert float(log[0]['grad_norm']) > 0
    # Loss should decrease overall (not necessarily every step but first > last generally)
    assert float(log[0]['loss']) > float(log[-1]['loss'])


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from tqdm import tqdm
import torch as t

def faded3_tqdm_grad_norm(n_steps=5):
    t.manual_seed(1)
    w = t.tensor([2.0], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=0.05)
    x = t.linspace(0, 1, 8)
    y_target = 3.0 * x
    log = []
    pbar = tqdm(range(n_steps), desc='GradWatch')
    for step in pbar:
        pred = w * x
        loss = ((pred - y_target) ** 2).mean()
        loss.backward()
        grad_norm = w.grad.abs().item()
        pf = dict(loss=f'{loss.item():.4f}', grad_norm=f'{grad_norm:.4f}')
        pbar.set_postfix(**pf)
        log.append(pf)
        optimizer.step()
        optimizer.zero_grad()
    return log
```
</details>